# 02. Train / Validation / Test Split & Preprocessing

## 목적
- Survival modeling cohort 529명을 Train / Validation / Test로 고정 분할
- PFI event 비율을 유지하도록 stratified split 수행
- Split 정보를 저장하여 이후 모든 모델에서 동일한 환자 분할 사용
- RNA 전처리는 Train 데이터 기준으로만 수행하여 data leakage 방지

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

SEED = 42

data_dir = Path("../data")

cohort_path = data_dir / "processed" / "kirc_modeling_cohort.parquet"
rna_path = data_dir / "processed" / "rna_tpm_modeling_raw.parquet"

cohort = pd.read_parquet(cohort_path)
rna_tpm = pd.read_parquet(rna_path)

print("Clinical cohort shape:", cohort.shape)
print("RNA TPM shape:", rna_tpm.shape)

Clinical cohort shape: (529, 10)
RNA TPM shape: (529, 19962)


In [2]:
print("환자 수:", len(cohort))
print("고유 patient_id:", cohort["patient_id"].nunique())

print("\nPFI event 분포")
print(cohort["pfi_event"].value_counts().sort_index())

print("\nPFI time")
print(cohort["pfi_time"].describe())

print("\nRNA 환자 수:", len(rna_tpm))
print("RNA index 고유 수:", rna_tpm.index.nunique())

# QC
assert len(cohort) == 529
assert cohort["patient_id"].nunique() == 529

assert cohort["pfi_event"].isin([0, 1]).all()
assert (cohort["pfi_time"] > 0).all()

assert rna_tpm.shape[0] == 529
assert rna_tpm.index.nunique() == 529

assert set(cohort["patient_id"]) == set(rna_tpm.index)

print("\n===== PRE-SPLIT QC PASSED =====")

환자 수: 529
고유 patient_id: 529

PFI event 분포
pfi_event
0.0    370
1.0    159
Name: count, dtype: int64

PFI time
count     529.000000
mean     1164.875236
std       945.494688
min         2.000000
25%       362.000000
50%      1014.000000
75%      1757.000000
max      4074.000000
Name: pfi_time, dtype: float64

RNA 환자 수: 529
RNA index 고유 수: 529

===== PRE-SPLIT QC PASSED =====


In [3]:
from sklearn.model_selection import train_test_split

SEED = 42

# 1차: Train 70% / 임시셋 30%
train_df, temp_df = train_test_split(
    cohort,
    test_size=0.30,
    random_state=SEED,
    stratify=cohort["pfi_event"]
)

# 2차: 임시셋을 Validation / Test로 절반씩 분할
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["pfi_event"]
)

print("Train      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))
print("Total      :", len(train_df) + len(val_df) + len(test_df))

Train      : 370
Validation : 79
Test       : 80
Total      : 529


In [4]:
def check_event_ratio(df, name):
    event_count = int(df["pfi_event"].sum())
    event_rate = df["pfi_event"].mean()

    print(
        f"{name:10s} | "
        f"N={len(df):3d} | "
        f"Event={event_count:3d} | "
        f"Event rate={event_rate:.3f}"
    )

check_event_ratio(train_df, "Train")
check_event_ratio(val_df, "Validation")
check_event_ratio(test_df, "Test")

Train      | N=370 | Event=111 | Event rate=0.300
Validation | N= 79 | Event= 24 | Event rate=0.304
Test       | N= 80 | Event= 24 | Event rate=0.300


In [5]:
splits = pd.concat([
    train_df[["patient_id"]].assign(split="train"),
    val_df[["patient_id"]].assign(split="validation"),
    test_df[["patient_id"]].assign(split="test")
], ignore_index=True)

splits.to_csv(
    "../data/processed/splits.csv",
    index=False
)

print(splits["split"].value_counts())
print("\n총 환자 수:", len(splits))
print("고유 환자 수:", splits["patient_id"].nunique())

split
train         370
test           80
validation     79
Name: count, dtype: int64

총 환자 수: 529
고유 환자 수: 529


,patient_id,split
0,TCGA-CJ-4900,train
1,TCGA-CW-6087,train
2,TCGA-BP-4326,train
3,TCGA-BP-4342,train
4,TCGA-B0-4836,train
...,...,...
524,TCGA-BP-5187,test
525,TCGA-G6-A8L8,test
526,TCGA-B0-5083,test
527,TCGA-BP-5181,test


In [8]:
splits_check = pd.read_csv("../data/processed/splits.csv")

print(splits_check["split"].value_counts())
print("총 환자 수:", len(splits_check))
print("고유 환자 수:", splits_check["patient_id"].nunique())

split
train         370
test           80
validation     79
Name: count, dtype: int64
총 환자 수: 529
고유 환자 수: 529


In [9]:
train_ids = splits_check.loc[
    splits_check["split"] == "train", "patient_id"
]

val_ids = splits_check.loc[
    splits_check["split"] == "validation", "patient_id"
]

test_ids = splits_check.loc[
    splits_check["split"] == "test", "patient_id"
]

rna_train = rna_tpm.loc[train_ids].copy()
rna_val = rna_tpm.loc[val_ids].copy()
rna_test = rna_tpm.loc[test_ids].copy()

print("RNA Train     :", rna_train.shape)
print("RNA Validation:", rna_val.shape)
print("RNA Test      :", rna_test.shape)

RNA Train     : (370, 19962)
RNA Validation: (79, 19962)
RNA Test      : (80, 19962)
